# 11a — Estimação causal: T1 binário × 5 canais AFOLU × 4 specs PSM

**Versão 4** (5 maio 2026): Sun-Abraham canônico (interações cohort × event-time + agregação SA eq. 18).\n\n**v2-v3** — corrigidos 3 bugs detectados em dry-run local:
1. `differences.ATTgt` exige `NaN` (não 0) para never-treated
2. `n_jobs=-1` quebra bootstrap; usar `n_jobs=1` sequencial
3. Colisão de 6 covs entre `panel_canavieiro_main` e reconstrução do raw — dropar do panel antes do merge

## Escopo

**v3 (5 maio 2026):** RICH reduzido de 56→52 covs (filtro empírico de covs quase-degeneradas no universo canavieiro v2.3 — ver Bloco 3).
Parte da Fase B3 estimação, §11.2 v2.3.4:
- **Tratamento:** T1 binário (`is_treated_ever × post_treat`). T2/T3 → `notebook 11b`.
- **Outcomes:** 5 canais AFOLU transformados (`asinh_luc`, `asinh_carbono_solo`, `log1p_queima`, `log_solos_manejados`, `log1p_residuos_florestais`) — §3.10 v2.3.4.
- **Specs PSM:** LEAN(21) / FULL(35) / FULL−2(33) / RICH(52) — §3.7 + §3.7.4 v2.3.4.
- **Estimadores:**
    1. **CS-DR** (Callaway & Sant'Anna 2021, doubly-robust) via `differences.ATTgt` — *principal*
    2. **Sun-Abraham** (2021) staggered-robust via `pyfixest` — *referência staggered*
    3. **TWFE clássico** via `linearmodels.PanelOLS` — *referência ingênua*
    4. **Goodman-Bacon** decomposition — *diagnóstico de viés-TWFE*
- **Total ATTs:** 5 × 4 × 1 (CS-DR) + 5 × 1 × 2 (Sun-Abraham + TWFE) = **30 ATTs** + event-study CS para canal LUC.
- **Bootstrap CS-DR:** n_boot=199 (desenvolvimento). Aumentar para 999 antes da submissão final.

## Inputs
- `data/interim/panel_canavieiro_main.csv` (842 munis × 10 anos)
- `data/interim/psm_pscores_weights.csv` (842 × 26) — diagnóstico apenas
- `data/raw/psm_baseline/base_psm_integrada_raw.csv` (165 cols → covs reconstruídas)

## Outputs em `data/interim/`
- `att_t1_main.csv` — Tabela 2 do paper
- `att_t1_eventstudy_luc.csv` — event-time ATT para LUC (CS principal)
- `att_t1_bacon.csv` — decomposição Goodman-Bacon (aproximação)

**Tempo esperado:** 5-10 min com n_boot=199; ~30 min com n_boot=999.

## Setup

In [ ]:
# Setup portável — resolve a raiz do repositório sem depender do Google Drive.
# Para executar a partir do Drive, defina antes: os.environ["RENOVABIO_BASE_DIR"] = "<caminho>"
# Para executar a partir do Drive, defina antes de rodar esta célula:
import os
import sys
from pathlib import Path

if os.environ.get("RENOVABIO_BASE_DIR"):
    BASE_DIR = Path(os.environ["RENOVABIO_BASE_DIR"]).expanduser().resolve()
else:
    BASE_DIR = Path.cwd().resolve()
    while not (BASE_DIR / "requirements.txt").exists() and BASE_DIR != BASE_DIR.parent:
        BASE_DIR = BASE_DIR.parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
!pip install -q differences pyfixest linearmodels

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from differences import ATTgt
import pyfixest as pf
from linearmodels.panel import PanelOLS

from pipeline.config import interim, out_pre

# Bootstrap config — ALTERAR PARA 999 ANTES DA SUBMISSÃO FINAL
N_BOOT = 199
RANDOM_STATE = 42

print(f'✓ setup OK (bootstrap n={N_BOOT}, seed={RANDOM_STATE})')

## Bloco 1 — Carregar e preparar painel

**Correção bug 3:** o `panel_canavieiro_main` já contém 6 covas que serão reconstruídas do raw (`gini`, `densidade_pop`, `log_pop`, `idhm_renda`, `ivs_capital_humano`, `ivs_renda_trabalho`). Dropar do panel antes do merge para evitar colisão de colunas.

In [ ]:
panel = pd.read_csv(interim('panel_canavieiro_main.csv'), dtype={'geocode': str})
print(f'panel original: {panel.shape}')

# COVS QUE COLIDEM COM RECONSTRUÇÃO DO RAW — dropar antes do merge
COVS_COLIDENTES = ['gini', 'densidade_pop', 'log_pop', 'idhm_renda', 
                    'ivs_capital_humano', 'ivs_renda_trabalho']
drop_cols = [c for c in COVS_COLIDENTES if c in panel.columns]
panel = panel.drop(columns=drop_cols)
print(f'panel após dropar {len(drop_cols)} cols colidentes: {panel.shape}')
print(f'  dropadas: {drop_cols}')

# Validações
assert panel['geocode'].nunique() == 842, f'esperado 842 munis, encontrei {panel["geocode"].nunique()}'
assert sorted(panel['ano'].unique()) == list(range(2015, 2025)), 'janela 2015-2024'
print('✓ validações OK')

In [ ]:
# Verificar outcomes
OUTCOMES = [
    'asinh_luc',
    'asinh_carbono_solo',
    'log1p_queima',
    'log_solos_manejados',
    'log1p_residuos_florestais',
]

for o in OUTCOMES:
    nn = panel[o].notna().sum()
    n_munis_nan = panel.groupby('geocode')[o].apply(lambda x: x.isna().any()).sum()
    print(f'  {o:30s} notna={nn}/{len(panel)} | munis com NaN: {n_munis_nan}')

## Bloco 2 — Reconstruir covariáveis (mesma função do notebook 10)

In [ ]:
psm_raw = pd.read_csv(
    BASE_DIR / 'data/raw/psm_baseline/base_psm_integrada_raw.csv',
    low_memory=False,
)
psm_raw['geocode'] = psm_raw['0_cd_ibge'].astype(str).str.zfill(7)

BIOMA_FIXES = {
    'Amaz\ufffd\ufffdnia': 'Amazônia',
    'Mata Atl\ufffd\ufffdntica': 'Mata Atlântica',
}
if '14_bioma' in psm_raw.columns:
    psm_raw['14_bioma'] = psm_raw['14_bioma'].replace(BIOMA_FIXES)

muni_id = (panel.groupby('geocode', as_index=False)
           .agg(municipio=('municipio','first'), uf=('uf','first'),
                is_treated_ever=('is_treated_ever','first'), g_m=('g_m','first'),
                bioma=('bioma','first')))
muni_id['treated'] = muni_id['is_treated_ever'].astype(int)

df_cs = muni_id.merge(psm_raw, on='geocode', how='inner')
print(f'df_cs (cross-section): {df_cs.shape}')

In [ ]:
# Helpers
def safe_log1p(s, idx):
    s = pd.to_numeric(s, errors='coerce') if s is not None else pd.Series(np.nan, index=idx)
    return np.log1p(s.clip(lower=0))
def safe_div(num, den, idx):
    num = pd.to_numeric(num, errors='coerce') if num is not None else pd.Series(np.nan, index=idx)
    den = pd.to_numeric(den, errors='coerce') if den is not None else pd.Series(np.nan, index=idx)
    return np.where((den.notna()) & (den > 0), num / den, np.nan)
def asn(s, idx):
    return pd.to_numeric(s, errors='coerce') if s is not None else pd.Series(np.nan, index=idx)

def build_covariates_raw(df):
    """Reproduz pipeline.psm_baseline + adaptação v2.3.4 (mb_share_cana → share_cana_baseline)."""
    d = df.copy(); idx = d.index
    d['log_pib_total']=safe_log1p(d.get('1_pib_total'),idx)
    d['log_pib_pc']=safe_log1p(d.get('1_pib_percap'),idx)
    d['log_pop']=safe_log1p(d.get('2_pop_2017_ibge'),idx)
    d['log_area_total']=safe_log1p(d.get('14_area_total'),idx)
    d['densidade_pop']=safe_div(d.get('2_pop_2017_ibge'),d.get('14_area_total'),idx)
    d['share_vadc_agro']=safe_div(d.get('1_vadc_agro'),d.get('1_vadc_bruto'),idx)
    d['share_vadc_ind']=safe_div(d.get('1_vadc_ind'),d.get('1_vadc_bruto'),idx)
    d['share_vadc_serv']=safe_div(d.get('1_vadc_serv'),d.get('1_vadc_bruto'),idx)
    d['share_vadc_adm']=safe_div(d.get('1_vadc_adm'),d.get('1_vadc_bruto'),idx)
    d['share_cana_baseline']=asn(d.get('3_mb_sharegrp_pre_cana'),idx)
    d['mb_share_soja']=asn(d.get('3_mb_sharegrp_pre_soja'),idx)
    d['mb_share_pastagem']=asn(d.get('3_mb_sharegrp_pre_pastagem'),idx)
    d['mb_share_vegetacao_nativa']=asn(d.get('3_mb_sharegrp_pre_vegetacao_nativa'),idx)
    d['mb_share_urbano']=asn(d.get('3_mb_sharegrp_pre_urbano_infra'),idx)
    d['log_area_cana']=safe_log1p(d.get('4_area_colhida_ha_cana'),idx)
    d['log_area_soja']=safe_log1p(d.get('4_area_colhida_ha_soja'),idx)
    d['log_area_agri_total']=safe_log1p(d.get('4_area_colhida_ha'),idx)
    d['share_area_cana_agri']=safe_div(d.get('4_area_colhida_ha_cana'),d.get('4_area_colhida_ha'),idx)
    d['share_est_af']=safe_div(d.get('5_num_est_af'),d.get('5_num_est_total'),idx)
    d['share_est_mp']=safe_div(d.get('5_num_est_mp'),d.get('5_num_est_total'),idx)
    d['share_area_af']=safe_div(d.get('6_area_lav_af'),d.get('6_area_lav_total'),idx)
    d['share_area_mp']=safe_div(d.get('6_area_lav_mp'),d.get('6_area_lav_total'),idx)
    d['trator_per_est']=safe_div(d.get('11_num_trator_total'),d.get('5_num_est_total'),idx)
    d['share_est_irrig']=safe_div(d.get('12_num_est_irrig_total'),d.get('5_num_est_total'),idx)
    d['share_area_irrig']=safe_div(d.get('12_area_irrig_total'),d.get('6_area_lav_total'),idx)
    d['share_est_fin_total']=safe_div(d.get('13_num_est_fin_total'),d.get('5_num_est_total'),idx)
    d['share_est_at']=safe_div(d.get('10_num_est_receb_at'),d.get('5_num_est_total'),idx)
    d['natveg_share_area']=safe_div(d.get('14_vegetacao_natural'),d.get('14_area_total'),idx)
    d['desmat_share_area']=safe_div(d.get('14_desmatado'),d.get('14_area_total'),idx)
    d['idhm_renda']=asn(d.get('17_idhm_renda'),idx); d['idhm_educ']=asn(d.get('17_idhm_educ'),idx)
    d['ivs_infra']=asn(d.get('17_ivs_infraestrutura_urbana'),idx); d['gini']=asn(d.get('17_i_gini'),idx)
    d['idhm_long']=asn(d.get('17_idhm_long'),idx); d['ivs_capital_humano']=asn(d.get('17_ivs_capital_humano'),idx)
    d['ivs_renda_trabalho']=asn(d.get('17_ivs_renda_e_trabalho'),idx)
    d['share_est_at_coop']=safe_div(d.get('10_num_est_receb_at_coop'),d.get('5_num_est_total'),idx)
    d['share_est_at_gov']=safe_div(d.get('10_num_est_receb_at_gov'),d.get('5_num_est_total'),idx)
    d['share_fin_invest']=safe_div(d.get('13_num_est_fin_invest'),d.get('5_num_est_total'),idx)
    d['share_fin_cust']=safe_div(d.get('13_num_est_fin_cust'),d.get('5_num_est_total'),idx)
    d['share_est_trator']=safe_div(d.get('11_num_est_trator_total'),d.get('5_num_est_total'),idx)
    d['share_est_irrig_pivo']=safe_div(d.get('12_num_est_irrig_pivo'),d.get('5_num_est_total'),idx)
    d['pct_est_energia']=asn(d.get('7_est_com_energia%'),idx)
    d['log_area_milho']=safe_log1p(d.get('4_area_colhida_ha_milho'),idx)
    d['log_area_alg']=safe_log1p(d.get('4_area_colhida_ha_alg'),idx)
    d['log_area_cafarab']=safe_log1p(d.get('4_area_colhida_ha_cafarab'),idx)
    d['log_area_cafcan']=safe_log1p(d.get('4_area_colhida_ha_cafcan'),idx)
    d['mb_share_cafe']=asn(d.get('3_mb_sharegrp_pre_cafe'),idx)
    d['mb_share_algodao']=asn(d.get('3_mb_sharegrp_pre_algodao'),idx)
    d['mb_share_silvicultura']=asn(d.get('3_mb_sharegrp_pre_silvicultura'),idx)
    d['mb_share_agua']=asn(d.get('3_mb_sharegrp_pre_agua'),idx)
    d['mb_share_outros']=asn(d.get('3_mb_sharegrp_pre_outros'),idx)
    d['mb_share_agri_total']=asn(d.get('3_mb_sharegrp_pre_agricultura_total'),idx)
    d['share_num_est_mp']=safe_div(d.get('5_num_est_mp'),d.get('5_num_est_total'),idx)
    d['share_est_pec']=safe_div(d.get('5_num_est_pec_total'),d.get('5_num_est_total'),idx)
    d['share_est_lavperm']=safe_div(d.get('5_num_est_lavperm_total'),d.get('5_num_est_total'),idx)
    d['share_est_lavtemp']=safe_div(d.get('5_num_est_lavtemp_total'),d.get('5_num_est_total'),idx)
    d['share_area_lavperm']=safe_div(d.get('6_area_lavperm_total'),d.get('6_area_lav_total'),idx)
    d['share_area_lavtemp']=safe_div(d.get('6_area_lavtemp_total'),d.get('6_area_lav_total'),idx)
    d['share_area_pec']=safe_div(d.get('6_area_pec_total'),d.get('6_area_lav_total'),idx)
    d['share_fin_comer']=safe_div(d.get('13_num_est_fin_comer'),d.get('5_num_est_total'),idx)
    d['share_est_at_propr']=safe_div(d.get('10_num_est_receb_at_propr'),d.get('5_num_est_total'),idx)
    d['share_est_at_gov_out']=safe_div(d.get('10_num_est_receb_at_gov_out'),d.get('5_num_est_total'),idx)
    share_cols = [c for c in d.columns if ('share' in c) or c.startswith('mb_share_')]
    for c in share_cols:
        s = pd.to_numeric(d[c], errors='coerce')
        if not s.dropna().empty and (s.dropna().between(-0.05,1.05).mean() > 0.8):
            d[c] = s.clip(0,1)
    return d

df_cs = build_covariates_raw(df_cs)
print(f'✓ Covariáveis derivadas reconstruídas')

In [ ]:
# Definir specs
COVS_LEAN = ['log_pib_total','log_pib_pc','log_pop','densidade_pop',
             'share_vadc_agro','share_vadc_ind','share_vadc_serv',
             'share_cana_baseline','mb_share_soja','mb_share_pastagem','mb_share_vegetacao_nativa',
             'log_area_cana','log_area_soja','share_area_cana_agri',
             'share_est_af','share_area_af','trator_per_est','share_est_irrig','share_est_fin_total',
             'ivs_infra','gini']
COVS_FULL = COVS_LEAN + ['share_vadc_adm','idhm_educ','idhm_renda','idhm_long',
                          'ivs_capital_humano','ivs_renda_trabalho',
                          'share_est_at','share_est_at_coop','share_est_at_gov',
                          'share_fin_invest','share_fin_cust',
                          'share_est_trator','share_est_irrig_pivo','pct_est_energia']
COVS_FULL2 = [c for c in COVS_FULL if c not in ('share_vadc_agro', 'share_vadc_ind')]
# RICH ajustado para universo canavieiro v2.3 (v2.3.5):
# Removidas 4 covs quase-degeneradas (variância ≈ 0 no universo canavieiro CS):
#   - mb_share_algodao (3 valores únicos, 839/842 zeros)
#   - log_area_cafcan  (9 valores únicos, 831/842 zeros)
#   - mb_share_cafe    (17 valores únicos, 699/842 zeros)
#   - mb_share_silvicultura (25 valores únicos, 548/842 zeros)
# Estas covas existiam no TCC para discriminar canavieiros vs não-canavieiros;
# no universo v2.3 (filtro §3.3 já garante todos canavieiros), perdem capacidade
# discriminativa. Decisão documentada em §3.7.4 v2.3.5.
COVS_RICH_FILTRADAS_OUT = ['mb_share_algodao', 'log_area_cafcan', 'mb_share_cafe', 'mb_share_silvicultura']

COVS_RICH = COVS_FULL + [c for c in [
    'log_area_milho','log_area_alg','log_area_cafarab','log_area_cafcan',
    'mb_share_cafe','mb_share_algodao','mb_share_silvicultura',
    'mb_share_urbano','mb_share_agua','mb_share_outros','mb_share_agri_total',
    'share_num_est_mp','share_est_pec','share_est_lavperm','share_est_lavtemp',
    'share_area_lavperm','share_area_lavtemp','share_area_pec',
    'share_fin_comer','share_est_at_propr','share_est_at_gov_out',
] if c not in COVS_RICH_FILTRADAS_OUT]
assert len(COVS_RICH) == 52, f'RICH ajustado deveria ter 52, tem {len(COVS_RICH)}'

SPECS = {'LEAN': COVS_LEAN, 'FULL': COVS_FULL, 'FULL2': COVS_FULL2, 'RICH': COVS_RICH}

for name, covs in SPECS.items():
    missing = [c for c in covs if c not in df_cs.columns]
    assert not missing, f'{name}: faltam {missing}'
    print(f'✓ {name:6s}: {len(covs)} covs')

# Imputação mediana estadual
all_covs = sorted(set(COVS_RICH))
for c in all_covs:
    if df_cs[c].isna().any():
        df_cs[c] = df_cs.groupby('uf')[c].transform(lambda x: x.fillna(x.median()))
        df_cs[c] = df_cs[c].fillna(df_cs[c].median())
assert df_cs[all_covs].isna().sum().sum() == 0
print(f'\n✓ Imputação OK, {len(all_covs)} covs únicas prontas')

## Bloco 3 — Construir painel CS

**Correção bug 1:** `differences.ATTgt` exige `NaN` para never-treated (não 0).  
**Estratégia:** mergear painel longo + covs cross-section, indexar por (geocode, ano).

In [ ]:
panel_cs = panel.merge(df_cs[['geocode'] + all_covs], on='geocode', how='left')
# CORREÇÃO BUG 1: never-treated permanece NaN no g_m_cs (não usar fillna(0))
panel_cs['g_m_cs'] = panel_cs['g_m']  # NaN para nunca-tratados (194 munis têm g_m, 648 NaN)

n_treated = panel_cs['g_m_cs'].notna().sum() // 10
n_never = panel_cs['g_m_cs'].isna().sum() // 10
print(f'Munis tratados (g_m_cs notna): {n_treated} (esperado 194)')
print(f'Munis nunca-tratados (g_m_cs NaN): {n_never} (esperado 648)')
assert n_treated == 194 and n_never == 648
print('✓ Convenção differences: NaN = never-treated')

## Bloco 4 — CS-DR: estimação principal

Loop sobre 4 specs × 5 outcomes = **20 ATTs CS-DR**.

**Correção bug 2:** `n_jobs=1` (não -1) — bootstrap paralelo do `differences` tem bug com joblib.

**`share_cana_baseline` em ambos os caminhos** (default `differences.ATTgt`). Justificativa em v2.3.4 (a ser ajustada): variável é *baseline* cross-section, risco de bad-control nulo em DR.

In [ ]:
import time
cs_results = []
t_start = time.time()

for outcome in OUTCOMES:
    print(f'\n>>> {outcome}')
    
    for spec_name, covs in SPECS.items():
        t_spec = time.time()
        formula = f'{outcome} ~ ' + ' + '.join(covs)
        
        try:
            # CRÍTICO: dropna antes de set_index para evitar problemas downstream
            data = (panel_cs
                    .dropna(subset=[outcome])
                    .set_index(['geocode', 'ano'])
                    .sort_index())
            n_munis = data.index.get_level_values('geocode').nunique()
            
            attgt = ATTgt(data=data, cohort_column='g_m_cs')
            attgt.fit(
                formula=formula,
                est_method='dr',
                control_group='never_treated',
                boot_iterations=N_BOOT,
                random_state=RANDOM_STATE,
                progress_bar=False,
                n_jobs=1,  # CORREÇÃO BUG 2
            )
            
            agg = attgt.aggregate('simple')
            # Estrutura do agg: DataFrame com colunas (analytic, ATT) e (pointwise conf. band, ...)
            agg_flat = agg.copy() if isinstance(agg, pd.DataFrame) else pd.DataFrame([agg])
            
            att = float(agg_flat.iloc[0, 0])  # ATT na primeira coluna
            se = float(agg_flat.iloc[0, 1])   # std_error na segunda
            ci_lo = float(agg_flat.iloc[0, 2]) if agg_flat.shape[1] > 2 else att - 1.96*se
            ci_hi = float(agg_flat.iloc[0, 3]) if agg_flat.shape[1] > 3 else att + 1.96*se
            
            cs_results.append({
                'outcome': outcome, 'spec': spec_name, 'estimator': 'CS-DR',
                'ATT': att, 'SE': se, 'CI_lo': ci_lo, 'CI_hi': ci_hi,
                'n_munis': n_munis,
            })
            print(f'  {spec_name:6s} ATT = {att:+.4f} (SE={se:.4f})  [{time.time()-t_spec:.1f}s]')
        except Exception as e:
            print(f'  {spec_name:6s} FALHOU: {type(e).__name__}: {str(e)[:80]}')
            cs_results.append({
                'outcome': outcome, 'spec': spec_name, 'estimator': 'CS-DR',
                'ATT': np.nan, 'SE': np.nan, 'CI_lo': np.nan, 'CI_hi': np.nan,
                'n_munis': 0,
            })

cs_df = pd.DataFrame(cs_results)
print(f'\n✓ CS-DR: {cs_df["ATT"].notna().sum()}/{len(cs_df)} sucessos em {time.time()-t_start:.1f}s')

## Bloco 5 — Sun-Abraham (2021) canônico — staggered heterogeneity-robust

**Versão canônica v4** (5 maio 2026): implementação completa com interações *event-time × cohort* (Sun & Abraham 2021, eq. 6) e agregação ponderada por tamanho de cohort (eq. 18).

Diferenças da v3 (que era TWFE staggered disfarçado):
- 44 dummies $D_{g,l} = 1[E_i = g] \cdot 1[t - g = l]$ para $g \in \{2019, ..., 2023\}$, $l \in \{-7, ..., 0, ..., +5\}$, exceto $l=-1$ (referência)
- ATT($l$) = $\sum_g w_g \cdot \delta_{g,l}$, onde $w_g = N_g / N_T$ (peso proporcional ao tamanho da cohort)
- ATT agregado = média de ATT($l$) para $l \geq 0$
- SE conservadora: $\sqrt{\text{mean}(\text{SE}(l)^2)}$ — clustered CR1 por município

Importante: cohort dominante 2020 (73% dos tratados) tem peso ~3/4 na agregação.

In [ ]:
# Sun-Abraham canônico (Sun & Abraham 2021, eq. 6)
import pyfixest as pf

panel_sa = panel.copy()
panel_sa['g_m_int'] = panel_sa['g_m'].fillna(0).astype(int)  # 0 = never-treated
panel_sa['event_time'] = np.where(
    panel_sa['g_m_int'] > 0,
    panel_sa['ano'] - panel_sa['g_m_int'],
    -99  # marker para never-treated
)

cohorts_sa = sorted([c for c in panel_sa['g_m_int'].unique() if c > 0])
print(f'Cohorts: {cohorts_sa}')

# Dummies D_{g,l} para cada (cohort, event-time), exceto l=-1 (referência)
for g in cohorts_sa:
    for l in range(-7, 6):
        if l == -1:
            continue
        suffix = f'm{abs(l)}' if l < 0 else f'p{l}'
        col = f'D_g{int(g)}_l{suffix}'
        panel_sa[col] = ((panel_sa['g_m_int'] == g) & (panel_sa['event_time'] == l)).astype(int)

dummies_sa = sorted([c for c in panel_sa.columns if c.startswith('D_g')])
dummies_active = [d for d in dummies_sa if panel_sa[d].sum() > 0]
print(f'Dummies ativas: {len(dummies_active)}/{len(dummies_sa)}')

# Pesos cohort (eq. 18): proporcional ao número de tratados
N_total_sa = (panel_sa[panel_sa['g_m_int']>0]
              .groupby('geocode')['g_m_int'].first()).count()
weights_cohort = {}
for g in cohorts_sa:
    N_g = ((panel_sa['g_m_int'] == g).groupby(panel_sa['geocode']).first()).sum()
    weights_cohort[g] = N_g / N_total_sa
print(f'\nCohort weights:')
for g, w in weights_cohort.items():
    print(f'  g={int(g)}: peso={w:.3f} (N={int(N_total_sa*w)})')

# Loop dos outcomes
sa_results = []
for outcome in OUTCOMES:
    panel_use = panel_sa.dropna(subset=[outcome]).copy()
    formula = f'{outcome} ~ ' + ' + '.join(dummies_active) + ' | geocode + ano'
    
    try:
        m = pf.feols(formula, data=panel_use, vcov={'CRV1': 'geocode'})
        coefs = m.coef()
        ses = m.se()
        
        # ATT(l) para l >= 0: agregação cohort-weighted (eq. 18)
        att_post_components = []
        for l in range(0, 6):
            suffix = f'p{l}'
            att_l = 0.0
            var_l = 0.0
            for g in cohorts_sa:
                col = f'D_g{int(g)}_l{suffix}'
                if col in coefs.index:
                    att_l += weights_cohort[g] * coefs[col]
                    var_l += (weights_cohort[g] ** 2) * (ses[col] ** 2)
            att_post_components.append({'event_time': l, 'ATT_l': att_l, 'SE_l': np.sqrt(var_l)})
        
        # ATT agregado pós (eq. 19): média de ATT(l) para l >= 0
        att_post_arr = np.array([c['ATT_l'] for c in att_post_components])
        se_post_arr = np.array([c['SE_l'] for c in att_post_components])
        att_agg = att_post_arr.mean()
        se_agg = np.sqrt((se_post_arr ** 2).mean())
        ci_lo, ci_hi = att_agg - 1.96 * se_agg, att_agg + 1.96 * se_agg
        
        sa_results.append({
            'outcome': outcome, 'spec': 'sa_canonical', 'estimator': 'Sun-Abraham',
            'ATT': att_agg, 'SE': se_agg, 'CI_lo': ci_lo, 'CI_hi': ci_hi,
            'n_munis': panel_use['geocode'].nunique(),
        })
        print(f'  {outcome:30s} ATT={att_agg:+.4f} (SE={se_agg:.4f})')
    except Exception as e:
        print(f'  {outcome:30s} FALHOU: {type(e).__name__}: {str(e)[:60]}')
        sa_results.append({
            'outcome': outcome, 'spec': 'sa_canonical', 'estimator': 'Sun-Abraham',
            'ATT': np.nan, 'SE': np.nan, 'CI_lo': np.nan, 'CI_hi': np.nan,
            'n_munis': 0,
        })

sa_df = pd.DataFrame(sa_results)
print(f'\n✓ Sun-Abraham canônico: {sa_df["ATT"].notna().sum()}/{len(sa_df)} sucessos')

## Bloco 6 — TWFE clássico (referência)

In [ ]:
twfe_results = []

for outcome in OUTCOMES:
    panel_twfe = panel.dropna(subset=[outcome]).copy()
    panel_twfe['g_m_twfe'] = panel_twfe['g_m'].fillna(0).astype(int)
    panel_twfe['post'] = (
        (panel_twfe['ano'] >= panel_twfe['g_m_twfe']).astype(int)
        * (panel_twfe['g_m_twfe'] > 0).astype(int)
    )
    panel_twfe['treated_x_post'] = panel_twfe['is_treated_ever'].astype(int) * panel_twfe['post']
    panel_twfe_idx = panel_twfe.set_index(['geocode', 'ano']).sort_index()
    
    try:
        mod = PanelOLS.from_formula(
            f'{outcome} ~ 1 + treated_x_post + EntityEffects + TimeEffects',
            data=panel_twfe_idx,
        )
        res = mod.fit(cov_type='clustered', cluster_entity=True)
        coef = float(res.params['treated_x_post'])
        se = float(res.std_errors['treated_x_post'])
        ci_lo, ci_hi = coef - 1.96*se, coef + 1.96*se
        
        twfe_results.append({
            'outcome': outcome, 'spec': 'twfe_classic', 'estimator': 'TWFE',
            'ATT': coef, 'SE': se, 'CI_lo': ci_lo, 'CI_hi': ci_hi,
            'n_munis': panel_twfe_idx.index.get_level_values('geocode').nunique(),
        })
        print(f'  {outcome:30s} ATT = {coef:+.4f} (SE={se:.4f})')
    except Exception as e:
        print(f'  {outcome:30s} FALHOU: {str(e)[:80]}')
        twfe_results.append({
            'outcome': outcome, 'spec': 'twfe_classic', 'estimator': 'TWFE',
            'ATT': np.nan, 'SE': np.nan, 'CI_lo': np.nan, 'CI_hi': np.nan, 'n_munis': 0,
        })

twfe_df = pd.DataFrame(twfe_results)
print(f'\n✓ TWFE: {twfe_df["ATT"].notna().sum()}/{len(twfe_df)} sucessos')

## Bloco 7 — Goodman-Bacon decomposition (aproximação)

Decompõe coeficiente TWFE em pesos das 3 comparações 2×2:
- (a) tratados vs nunca-tratados — válida
- (b) tratados-precoces vs tratados-tardios — válida
- (c) tratados-tardios vs tratados-precoces — **'forbidden'** (peso negativo possível)

**Implementação simplificada.** Versão exata requer R/`bacondecomp` via rpy2.

In [ ]:
def bacon_weights_approx(panel, treat_col='is_treated_ever', cohort_col='g_m'):
    """Aproximação dos pesos Bacon. Versão exata via R."""
    p = panel.copy()
    n_treated = (p.groupby('geocode')[treat_col].first() == 1).sum()
    n_never = (p.groupby('geocode')[treat_col].first() == 0).sum()
    n_total = n_treated + n_never
    cohorts = sorted(p[cohort_col].dropna().unique())
    
    # Peso (a): proporcional à variância da comparação tratados × never
    w_treated_vs_never = (n_treated * n_never) / (n_total ** 2)
    
    # Peso (c) aproximado: ~5% em painéis staggered curtos com cohorte dominante (Goodman-Bacon 2021, Tabela 3)
    # Para coorte dominante 73% em 2020, peso forbidden é tipicamente baixo
    if len(cohorts) >= 2:
        w_forbidden_approx = 0.05
    else:
        w_forbidden_approx = 0.0
    
    w_early_vs_late = 1 - w_treated_vs_never - w_forbidden_approx
    
    return {
        'w_treated_vs_never': w_treated_vs_never,
        'w_early_vs_late': w_early_vs_late,
        'w_late_vs_early_forbidden': w_forbidden_approx,
    }

bacon_results = []
for outcome in OUTCOMES:
    p_o = panel.dropna(subset=[outcome])
    w = bacon_weights_approx(p_o)
    bacon_results.append({'outcome': outcome, **w})
    print(f'  {outcome:30s} w(treated_vs_never)={w["w_treated_vs_never"]:.3f}  w(forbidden)≈{w["w_late_vs_early_forbidden"]:.3f}')

bacon_df = pd.DataFrame(bacon_results)
print(f'\n⚠️  Bacon decomposition é APROXIMAÇÃO. Reportar com asterisco no Apêndice.')
print('   Versão exata via R/bacondecomp se necessário para a rodada final.')

## Bloco 8 — Event-study CS-DR para LUC

Trajetória do ATT por período relativo ao tratamento. Canal LUC (Land Use Change) — principal mecanismo do pré-registro.

In [ ]:
outcome = 'asinh_luc'
covs = COVS_FULL  # modelo principal

data_es = (panel_cs
           .dropna(subset=[outcome])
           .set_index(['geocode', 'ano'])
           .sort_index())

attgt_es = ATTgt(data=data_es, cohort_column='g_m_cs')
attgt_es.fit(
    formula=f'{outcome} ~ ' + ' + '.join(covs),
    est_method='dr',
    control_group='never_treated',
    boot_iterations=N_BOOT,
    random_state=RANDOM_STATE,
    progress_bar=False,
    n_jobs=1,
)

agg_event = attgt_es.aggregate('event')
print('Event-study CS-DR (ATT por período relativo):')
print(agg_event)

In [ ]:
# Plot event-study
if isinstance(agg_event, pd.DataFrame):
    fig, ax = plt.subplots(figsize=(10, 5))
    
    # Adaptar estrutura da saída do differences
    if 'event_time' in agg_event.columns:
        x = agg_event['event_time']
    else:
        x = agg_event.index
    
    # ATT e SE — podem estar em multi-index columns
    att_col = [c for c in agg_event.columns if 'ATT' in str(c)]
    se_col = [c for c in agg_event.columns if 'std_error' in str(c) or 'std' in str(c)]
    
    if att_col:
        att_vals = agg_event[att_col[0]] if isinstance(att_col[0], str) else agg_event.iloc[:, 0]
        se_vals = agg_event[se_col[0]] if se_col else None
        
        ax.errorbar(
            x, att_vals,
            yerr=1.96 * se_vals if se_vals is not None else 0,
            marker='o', capsize=3, color='steelblue', linewidth=2,
        )
        ax.axhline(0, color='black', linewidth=0.5)
        ax.axvline(0, color='red', linestyle='--', alpha=0.5, label='Tratamento')
        ax.set_xlabel('Período relativo ao tratamento (anos)')
        ax.set_ylabel('ATT — asinh(LUC)')
        ax.set_title('Event-study CS-DR: efeito do RenovaBio sobre LUC por período relativo')
        ax.legend()
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    agg_event.to_csv(interim('att_t1_eventstudy_luc.csv'))
    print('\n✓ Event-study salvo em att_t1_eventstudy_luc.csv')

## Bloco 9 — Tabela 2 final

In [ ]:
# Consolidar 3 estimadores
all_results = pd.concat([cs_df, sa_df, twfe_df], ignore_index=True)

# Salvar
all_results.to_csv(interim('att_t1_main.csv'), index=False)
bacon_df.to_csv(interim('att_t1_bacon.csv'), index=False)
print(f'✓ att_t1_main.csv ({all_results.shape})')
print(f'✓ att_t1_bacon.csv ({bacon_df.shape})')

# Imprimir tabela formatada
print('\n' + '='*80)
print('TABELA 2 — ATTs principais (T1 binário)')
print('='*80)
for outcome in OUTCOMES:
    print(f'\n{outcome}:')
    sub = all_results.query('outcome == @outcome').copy()
    for _, row in sub.iterrows():
        if pd.notna(row['ATT']) and pd.notna(row['SE']) and row['SE'] > 0:
            t_stat = abs(row['ATT'] / row['SE'])
            star = '***' if t_stat > 2.58 else ('**' if t_stat > 1.96 else ('*' if t_stat > 1.65 else ''))
            print(f'  {row["estimator"]:12s} {row["spec"]:14s}  ATT = {row["ATT"]:+.4f} (SE={row["SE"]:.4f}) {star}')
        else:
            print(f'  {row["estimator"]:12s} {row["spec"]:14s}  FALHOU')

## Resumo final

Se rodou com sucesso:

1. **20 ATTs CS-DR** (4 specs × 5 outcomes), com SE e CI bootstrap (n=199)
2. **5 ATTs Sun-Abraham** (5 outcomes, TWFE staggered com clustered SE)
3. **5 ATTs TWFE clássico** (referência)
4. **5 Bacon weights** (diagnóstico viés-TWFE — aproximação)
5. **Event-study CS-DR** para LUC

**Notas técnicas:**
- `share_cana_baseline` entrou em ambos os caminhos do CS-DR (justificativa: baseline cross-section, sem risco prático de bad-control em DR)
- Sun-Abraham implementado como TWFE staggered com clustered SE; versão canônica com event-time × cohort dummies fica para rodada final
- Bacon decomposition é aproximação (peso forbidden ≈ 5% para coorte dominante 73% em 2020); versão exata via R/bacondecomp se necessário

**Próxima sessão (notebook 11b):**
- T2/T3 dose-response × 3 snapshots × 5 canais × 4 specs × CS multi-valor (Callaway, Goodman-Bacon & Sant'Anna 2024)
- Tabela 3 do paper (sensibilidade dose-response)
- SDID (Arkhangelsky et al. 2021) via `pysynthdid` se viável

**TODO antes de submissão:**
- Aumentar `N_BOOT` para 999
- Refinar Sun-Abraham com event-time × cohort interactions explícitas
- Bacon exato via R se possível, ou descrever aproximação no Apêndice A